# Notebook 02 — Exploratory Data Analysis

Measures the dataset prepared by notebook 01 and writes the two sets of numbers the training notebook reloads: the per-channel normalisation statistics and the class weights.

Statistics that only describe the data are reported for every split. Statistics that become training parameters are measured on the **training split alone**, so no information from validation or test reaches the model.

All the logic lives in `src/exploration.py`; this notebook imports and calls.

# 0. Environment Setup

Mounts Google Drive on Colab, makes the shared modules in `src/` importable and seeds the random number generators.

Unlike notebook 01, nothing here creates folders or writes to `data/`: this notebook reads what notebook 01 produced. Run notebook 01 first if the dataset has changed.

In [ ]:
# =====================================================
# 0.1 - BOOTSTRAP: DRIVE MOUNT AND MODULE PATH
# =====================================================
import importlib
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/VineProject_ToF")
LOCAL_ROOT = Path("/content/VineProject_ToF")

if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # Reading from Drive goes through FUSE and is slow on small files, and the
    # Dataset opens five per sample. A copy on the session disk is used when
    # one is there; without it nothing changes and Drive is read directly.
    PROJECT_ROOT = LOCAL_ROOT if (LOCAL_ROOT / "src" / "config.py").exists() else DRIVE_ROOT
    print(f"Project root: {PROJECT_ROOT}"
          f"{'  (session disk)' if PROJECT_ROOT == LOCAL_ROOT else '  (Drive)'}")
else:
    PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "src" / "config.py").exists():
    raise FileNotFoundError(f"config.py not found under: {PROJECT_ROOT / 'src'}")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import config


def reload_project_modules() -> None:
    """Reload the modules in src/ after editing them.

    Python caches imported modules, so a module edited on disk is not picked up
    by a simple re-import. Run this cell after changing config.py,
    data_preparation.py or exploration.py, then re-run the cells that import
    from them.
    """
    # config first, then data_preparation: each reads the previous one's
    # constants at import time.
    for module_name in ("config", "data_preparation", "exploration"):
        if module_name in sys.modules:
            importlib.reload(sys.modules[module_name])
            print(f"  reloaded: {module_name}")


# Figures are written here rather than into an experiment folder: they describe
# the dataset, not a training run, so they stay valid across every run.
FIGURES_DIR = config.RESULTS_DIR / "figures" / "eda"

config.print_environment_info()
config.set_seed()
print(f"Figures will be saved to: {FIGURES_DIR}")

# 1. Dataset Inventory

Checks that every sample is complete across all six sources — the three raw modalities and the three maps derived from the annotations — and reports the format each source is loaded in.

Files are read through `read_modality_image()`, so the report describes the arrays the `Dataset` will receive rather than the bytes on disk. Part of the raw files store the single-channel modalities as three identical planes; that difference is reconciled at read time.

The observed value range doubles as a validity check on the label maps: multiclass must stay within 0–4, binary within 0–1, skeleton within the classes being skeletonised.

Input: `data/raw/` and `data/processed/`

In [ ]:
# =====================================================
# 1.1 - CONSISTENCY AND FORMAT
# =====================================================
from exploration import check_dataset_consistency, describe_image_formats

check_dataset_consistency()
IMAGE_FORMATS = describe_image_formats()

In [ ]:
# =====================================================
# 1.2 - WHAT THE DATA LOOKS LIKE
# =====================================================
from exploration import show_samples

show_samples(n_samples=3, save_path=FIGURES_DIR / "samples.png")

# 2. Class Distribution

Two quantities per split. The **pixel share** drives the class weights and shows how imbalanced the problem is. The **image presence** — how many images a class appears in at all — answers a question the share cannot: a class missing from a partition leaves its IoU undefined there, which silently makes the mean IoU incomparable between runs.

Label maps are read from `data/processed/` filtered by the split, not from the augmented folders: a flip permutes pixels without changing their values, so the augmented copies carry identical counts at four times the reading cost.

Output: `results/figures/eda/class_distribution.png`

In [ ]:
# =====================================================
# 2.1 - PIXEL SHARE AND CLASS PRESENCE
# =====================================================
from exploration import compute_class_distribution, plot_class_distribution

DISTRIBUTION = compute_class_distribution()
plot_class_distribution(DISTRIBUTION, save_path=FIGURES_DIR / "class_distribution.png")

# 3. Channel Statistics

Per-channel mean and standard deviation, used to normalise the model input.

Every modality declared in `config.MODALITY_CHANNELS` is measured, whether or not it is currently enabled: the statistics describe the data, not the experiment, so re-enabling a modality later can never find a file that lacks its channels. Channels follow `config.MODALITY_ORDER`, the order the `Dataset` will stack them in — and the order OpenCV returns, which is BGR.

Only the **training** values are saved. The other splits are descriptive: a marked difference between them would point at drift between acquisition batches.

Output: `data/processed/channel_stats.json`, `results/figures/eda/channel_histograms.png`

In [ ]:
# =====================================================
# 3.1 - MEASURE THE CHANNELS
# =====================================================
from exploration import compute_channel_statistics, plot_channel_histograms

CHANNEL_STATISTICS = compute_channel_statistics()
plot_channel_histograms(
    CHANNEL_STATISTICS, save_path=FIGURES_DIR / "channel_histograms.png"
)

In [ ]:
# =====================================================
# 3.2 - SAVE THE TRAINING STATISTICS
# =====================================================
from data_preparation import get_splits_hash

SPLITS_HASH = get_splits_hash()

config.save_channel_stats(
    means=CHANNEL_STATISTICS["train"]["means"],
    stds=CHANNEL_STATISTICS["train"]["stds"],
    splits_hash=SPLITS_HASH,
)

# 4. Class Weights

Turns the training frequencies into the weights the loss applies to each class. Without them the loss is dominated by Background, and a model predicting Background everywhere already scores well.

`sqrt_inverse` is the default because `inverse` produces a ratio above 300 between the heaviest and lightest class, which makes training unstable on the rare ones. Note that with normalisation enabled `median_freq` is not a distinct method: the median cancels when every weight is divided by the smallest, so it returns exactly what `inverse` returns.

Output: `data/processed/class_weights.json`, `results/figures/eda/class_weights.png`

In [ ]:
# =====================================================
# 4.1 - COMPUTE AND COMPARE
# =====================================================
from exploration import compute_class_weights, plot_class_weight_methods

CLASS_WEIGHTS = compute_class_weights(DISTRIBUTION["train"]["pixel_frequencies"])
plot_class_weight_methods(
    DISTRIBUTION["train"]["pixel_frequencies"],
    save_path=FIGURES_DIR / "class_weights.png",
)

In [ ]:
# =====================================================
# 4.2 - SAVE THE WEIGHTS
# =====================================================
config.save_class_weights(
    weights=CLASS_WEIGHTS,
    method=config.CLASS_WEIGHT_CONFIG["method"],
    splits_hash=SPLITS_HASH,
)

# 5. Skeleton Coverage

Measures the share of each class occupied by its own dilated centreline. A high value means the tube has swallowed the structure, at which point clDice stops measuring whether the centreline is connected and turns into an area-overlap measure — losing the property it is adopted for.

This is the analysis behind keeping `dilation_kernel_size` at 2 rather than moving to the symmetric 3, made reproducible. It runs on every image: it describes a property of the annotations and justifies a configuration choice, it does not produce a training parameter.

Skeletons are rebuilt in memory for each kernel, so the files written by notebook 01 are neither read for this nor overwritten.

Output: `results/figures/eda/skeleton_coverage.png`, `results/figures/eda/skeletons.png`

In [ ]:
# =====================================================
# 5.1 - KERNEL 2 VERSUS KERNEL 3
# =====================================================
from exploration import compute_skeleton_coverage, plot_skeleton_coverage

COVERAGE = compute_skeleton_coverage()
plot_skeleton_coverage(COVERAGE, save_path=FIGURES_DIR / "skeleton_coverage.png")

In [ ]:
# =====================================================
# 5.2 - SKELETONS AS THEY WERE WRITTEN
# =====================================================
from exploration import show_skeleton_examples

show_skeleton_examples(n_samples=3, save_path=FIGURES_DIR / "skeletons.png")

# 6. Offline Augmentation

Visual check on what notebook 01 materialised. Each training sample is shown next to the flipped copies written beside it, image above and label below.

A flip applied to the image but not to its label would be obvious here and invisible in any metric until the model failed to learn — which is why this check reads from `data/augmented/train/` rather than applying a transformation on the fly.

The online augmentation used during training is a different pipeline, visualised in notebook 03 once the `Dataset` class exists.

Output: `results/figures/eda/offline_augmentation.png`

In [ ]:
# =====================================================
# 6.1 - FLIPS AS THEY WERE WRITTEN
# =====================================================
from exploration import show_offline_augmentation

show_offline_augmentation(
    n_samples=2, save_path=FIGURES_DIR / "offline_augmentation.png"
)